# Unsupervised Learning Practical
### Customer Segmentation of a Pune Retail Chain using K-Means and Kernel K-Means

**Goal of this practical:**
A retail chain in **Pune** has data about its customers -- age, income, spending behaviour, number of visits, average bill and online orders.

But there is **no answer column** in this dataset. Nobody has ever labelled these customers as "Student", "Rich Family" or "Budget Buyer".

So the management asks:

1. *"Can the computer find natural groups of customers on its own?"* -> **K-Means (Clustering)**
2. *"And what if those groups are not simple round blobs, but curved or ring shaped?"* -> **Kernel K-Means (Non-linear Clustering)**


Here no answer is given at all. The algorithm must discover the pattern by itself -- that is **Unsupervised Learning**.

| | Supervised (SVM practical) | Unsupervised (this practical) |
|---|---|---|
| Data given | X (features) **and** y (answer) | Only X (features) |
| Task | Predict y for a new sample | Find hidden structure in X |
| Example | Predict house price from area | Group customers into segments |
| Measured by | R2 / RMSE against true y | Inertia / Silhouette (no true y exists) |

### The two algorithms we implement

| Algorithm | Type | Core Idea |
|---|---|---|
| **K-Means** (Part A) | Clustering -- **linear** | Places K centers in the data and repeatedly puts every customer into the group of the **nearest center**, then moves each center to the average of its members. Boundaries are **straight lines**. |


Notice the pattern: **K-Means -> Kernel K-Means** is the same upgrade as **Linear SVM -> RBF SVM** in the previous practical. One single idea -- the **kernel trick** -- is reused in both.

Both algorithms are run on the **same** `pune_customers.csv` file, and in both parts you can **enter a new customer's details yourself**.

## Step 1: Import the Libraries

We need only 5 imports for this whole practical. Nothing complicated.

| Import | Why we need it |
|---|---|
| `pandas` | to read the CSV file and work with tables |
| `numpy` | for basic number/array maths |
| `matplotlib.pyplot` | to draw the graphs |
| `StandardScaler` | to bring all columns to the same scale |
| `KMeans` | the clustering algorithm (Part A) |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2: Load the Dataset

We load `pune_customers.csv`, which has these columns:

- **Customer_ID** -> unique id of the customer (only for reference, not used in maths)
- **Locality** -> area in Pune (Kothrud, Baner, Hinjewadi ...) -- a text column
- **Age** -> age of the customer in years
- **Annual_Income_Lakhs** -> yearly income in Lakhs of Rupees
- **Spending_Score** -> a 1 to 100 score given by the store based on how much the customer spends
- **Visits_Per_Month** -> how many times the customer visits the store in a month
- **Avg_Bill_Rs** -> average bill amount per visit in Rupees
- **Online_Orders_Per_Month** -> number of orders placed through the mobile app

**Note:** there is **no "Segment" or "Type" column** here. Finding that segment is exactly our job.

In [2]:
data = pd.read_csv("pune_customers.csv")

print("Shape of dataset (rows, columns):", data.shape)
data.head(10)

Shape of dataset (rows, columns): (250, 8)


,Customer_ID,Locality,Age,Annual_Income_Lakhs,Spending_Score,Visits_Per_Month,Avg_Bill_Rs,Online_Orders_Per_Month
0,PUN1000,Wagholi,70,6.0,28,3,1145,0
1,PUN1001,Hinjewadi,32,12.8,84,7,2737,10
2,PUN1002,Hadapsar,47,21.2,81,5,7025,5
3,PUN1003,Deccan,38,30.2,73,4,9152,7
4,PUN1004,Wagholi,60,4.8,25,1,1600,2
5,PUN1005,Kothrud,24,4.5,76,7,478,12
6,PUN1006,Hadapsar,21,3.9,60,6,881,16
7,PUN1007,Hinjewadi,22,1.1,54,11,885,17
8,PUN1008,Kalyani Nagar,27,8.0,69,5,2897,16
9,PUN1009,Shivajinagar,44,24.3,59,4,6902,7


In [3]:
print(data.describe().round(2))

# missing values
print("\nMissing values in each column:")
print(data.isnull().sum())

          Age  Annual_Income_Lakhs  Spending_Score  Visits_Per_Month  \
count  250.00               250.00          250.00            250.00   
mean    40.50                15.66           54.09              5.06   
std     15.52                11.26           26.71              2.81   
min     18.00                 1.10            8.00              1.00   
25%     26.00                 6.02           27.00              3.00   
50%     41.00                11.85           63.00              5.00   
75%     51.00                25.88           78.00              7.00   
max     72.00                39.70           92.00             11.00   

       Avg_Bill_Rs  Online_Orders_Per_Month  
count       250.00                   250.00  
mean       2859.09                     6.96  
std        2366.34                     5.62  
min         381.00                     0.00  
25%         967.25                     2.00  
50%        2336.50                     5.00  
75%        3467.75           

## Step 3: Select the Features and Scale Them

We drop `Customer_ID` (just a name) and `Locality` (text, not a number) and keep the **6 numeric columns**.

### Why is scaling compulsory?

Look at the ranges printed above:

- `Avg_Bill_Rs` goes up to about **9500**
- `Spending_Score` goes only up to **92**
- `Visits_Per_Month` goes only up to about **11**

Both K-Means and PCA work on **distance / spread**:

`distance = sqrt( (bill1 - bill2)^2 + (score1 - score2)^2 + ... )`

A difference of 2000 in bill amount would completely crush a difference of 40 in spending score. The model would then cluster only on bill amount and ignore everything else.

**StandardScaler** converts every column using `z = (x - mean) / standard_deviation`, so each column gets **mean 0 and standard deviation 1**, and all 6 columns get an **equal vote**.

(This is the same `StandardScaler` we used before training SVR in the SVM practical.)

In [4]:
# 6 numeric columns for satandard sacle
features = ["Age", "Annual_Income_Lakhs", "Spending_Score",
            "Visits_Per_Month", "Avg_Bill_Rs", "Online_Orders_Per_Month"]

X = data[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Done Standard Scaler")

Done Standard Scaler


### Step A2: Confirming K --5

### Step A3: Train the Final K-Means Model

In [5]:
K = 5   

kmeans = KMeans(n_clusters=K, 
                init="k-means++",      #k-means++ improve centriod in each cluster (dynamically updated)
                n_init=10, 
                random_state=42) 

clusters = kmeans.fit_predict(X_scaled)

# add the cluster number as a new column 
data["Cluster"] = clusters

print("K-Means trained successfully!")
print("Final Inertia:", round(kmeans.inertia_, 2))
print("Iterations taken to converge:", kmeans.n_iter_) #iterations

print("\nNumber of customers in each cluster:")
print(data["Cluster"].value_counts().sort_index())

/lib/python3.14/site-packages/threadpoolctl.py:1135: RuntimeWarning: JsProxy.as_object_map() is deprecated. Use as_py_json() instead.
  for filepath in LDSO.loadedLibsByName.as_object_map():


K-Means trained successfully!
Final Inertia: 184.25
Iterations taken to converge: 10

Number of customers in each cluster:
Cluster
0    55
1    50
2    50
3    50
4    45
Name: count, dtype: int64


### Step A4: Cluster Profiling -- what do these clusters MEAN?

All Cluster View


In [6]:
# centroids we convert them back to real units table without cust
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_table = pd.DataFrame(centroids.round(1), columns=features)
centroid_table.index.name = "Cluster"

print("CLUSTER CENTROIDS IN ORIGINAL UNITS\n")
print(centroid_table.to_string())

CLUSTER CENTROIDS IN ORIGINAL UNITS

          Age  Annual_Income_Lakhs  Spending_Score  Visits_Per_Month  Avg_Bill_Rs  Online_Orders_Per_Month
Cluster                                                                                                   
0        28.9                 11.4            80.2               7.1       2415.0                     11.7
1        48.0                 29.3            21.1               2.5       2971.5                      2.5
2        44.1                 27.3            73.8               4.6       7002.1                      4.7
3        20.8                  3.0            66.3               8.5        638.4                     13.9
4        64.2                  6.8            23.4               2.2       1141.0                      1.0


### Step A6: Predict the Segment of a NEW Customer (User Input)

A new customer walks into the store. K-Means can immediately tell which segment they belong to, by finding the **nearest centroid**.

**Important:** the new customer must be scaled using `scaler.transform()` with the **same scaler that was fitted on the training data**. We never fit a new scaler, otherwise the numbers would be on a different scale than the centroids and the answer would be wrong.

In [7]:
age     = float(input("Enter Age                     : "))
income  = float(input("Enter Annual Income (Lakhs)   : "))
spend   = float(input("Enter Spending Score (1-100)  : "))
visits  = float(input("Enter Visits Per Month        : "))
bill    = float(input("Enter Average Bill (Rs)       : "))
online  = float(input("Enter Online Orders Per Month : "))

new_customer = pd.DataFrame([[age, income, spend, visits, bill, online]],
                            columns=features)

new_scaled = scaler.transform(new_customer)
predicted_cluster = kmeans.predict(new_scaled)[0]

print("\nThis customer belongs to Cluster", predicted_cluster)

Enter Age                     :  4
Enter Annual Income (Lakhs)   :  4
Enter Spending Score (1-100)  :  4
Enter Visits Per Month        :  4
Enter Average Bill (Rs)       :  4
Enter Online Orders Per Month :  44



This customer belongs to Cluster 3
